In [6]:
import java.io.File

data class Ten(
    val first: String,
    val second: List<String>,
    val third: IntArray
)

val input = File("10.txt").readLines().map { line ->
    val parts = line.split(" ")
    val first = parts.first()
        .removeSurrounding("[", "]")
        .replace('.', '0')
        .replace('#', '1')
    val second = parts.drop(1).dropLast(1)
        .map { tuple ->
            tuple
                .removeSurrounding("(", ")")
                .split(",")
                .map { it.toInt() }
                .let { indices ->
                    val maxIndex = first.length - 1
                    (0..maxIndex).joinToString("") { index ->
                        if (index in indices) "1" else "0"
                    }
                }
        }
    val third = parts.last()
        .removeSurrounding("{", "}")
        .split(",")
        .map { it.toInt() }
        .toIntArray()

    Ten(first, second, third)
}

In [7]:
// https://github.com/Z3Prover/z3/releases
// put into a folder "z3"
// there should be the jar and the lib folder inside z3
@file:DependsOn("z3/com.microsoft.z3.jar")

// Add the file path of the bin to your system's PATH
// kinda stupid that this question needs z3

In [9]:
import com.microsoft.z3.*

infix fun String.xorBinary(other: String): String {
    val maxLength = maxOf(this.length, other.length)
    val paddedThis = this.padStart(maxLength, '0')
    val paddedOther = other.padStart(maxLength, '0')

    return paddedThis.zip(paddedOther) { a, b ->
        if (a != b) '1' else '0'
    }.joinToString("")
}

fun findCombination(expected: String, options: List<String>): List<String> {
    val start = "0".repeat(expected.length)
    val queue = ArrayDeque<Pair<String, List<String>>>()
    queue.add(start to emptyList())
    val visited = mutableSetOf<String>()
    visited.add(start)

    while (queue.isNotEmpty()) {
        val (current, path) = queue.removeFirst()
        for (option in options) {
            val next = current xorBinary option
            if (next == expected) {
                return path + option
            }
            if (next !in visited) {
                visited.add(next)
                queue.add(next to (path + option))
            }
        }
    }
    error("")
}

val part1 = input.sumOf { question ->
    findCombination(question.first, question.second).size
}

// I'm not too particular sure what z3 actually do, aside from optimising an linear programming question
fun solvePart2WithZ3(targets: IntArray, options: List<String>): Int {
    val dims = targets.size
    val buttons = options.map { binaryStr ->
        binaryStr.foldIndexed(0L) { idx, acc, c ->
            if (c == '1') acc or (1L shl idx) else acc
        }
    }

    val config = mutableMapOf("model" to "true")
    Context(config).use { ctx ->
        val buttonPresses = buttons.indices.associate { idx ->
            idx to ctx.mkIntConst("p$idx")
        }
        val totalPresses = ctx.mkAdd(*buttonPresses.values.toTypedArray())
        val opt = ctx.mkOptimize()
        for (j in 0 until dims) {
            val affectingPresses = buttonPresses.filter { (idx, _) ->
                (buttons[idx] ushr j) and 1L == 1L
            }.values

            if (affectingPresses.isNotEmpty()) {
                val sum = ctx.mkAdd(*affectingPresses.toTypedArray())
                opt.Add(ctx.mkEq(sum, ctx.mkInt(targets[j])))
            } else {
                if (targets[j] != 0) {
                    return Int.MAX_VALUE
                }
            }
        }
        for (pressVar in buttonPresses.values) {
            opt.Add(ctx.mkGe(pressVar, ctx.mkInt(0)))
        }
        val handle = opt.MkMinimize(totalPresses)
        if (opt.Check() == Status.SATISFIABLE) {
            return (handle.value as IntNum).int
        } else {
            return Int.MAX_VALUE
        }
    }
}

val part2 = input.sumOf { question ->
    solvePart2WithZ3(question.third, question.second)
}